In [1]:
! pip install -q --upgrade langchain langchain-openai langchain-core langchain_community docx2txt pypdf  langchain_chroma sentence_transformers

In [2]:
import langchain
print(langchain.__version__)

0.3.23


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
langchain_api_key = os.getenv("LANGCHAIN_API_KEY")
langchain_tracing = os.getenv("LANGCHAIN_TRACING_V2")
langchain_project = os.getenv("LANGCHAIN_PROJECT")

os.environ["OPENAI_API_KEY"] = openai_api_key
os.environ["LANGCHAIN_API_KEY"] = langchain_api_key
os.environ["LANGCHAIN_TRACING_V2"] = langchain_tracing
os.environ["LANGCHAIN_PROJECT"] = langchain_project

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not set in .env")


###Call LLM

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
llm_response = llm.invoke("Hi")

llm_response

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 8, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_80cf447eee', 'id': 'chatcmpl-BMIfwRbCXUEgsUyJaXX9alOhKTaLR', 'finish_reason': 'stop', 'logprobs': None}, id='run-d88e9d0c-148a-40fc-beff-619114d86e3e-0', usage_metadata={'input_tokens': 8, 'output_tokens': 10, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

###Parsing Output

In [5]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
output_parser.invoke(llm_response)

'Hello! How can I assist you today?'

###Simple Chain

In [6]:
chain = llm | output_parser
chain.invoke("Tell me about RAG")

'RAG can refer to several different concepts depending on the context. Here are a few common interpretations:\n\n1. **RAG (Red, Amber, Green) Status Reporting**: In project management and reporting, RAG is a visual way to represent the status of a project or its components using colors:\n   - **Red** indicates significant issues that need immediate attention.\n   - **Amber** (or yellow) indicates caution and potential problems that may arise if not addressed.\n   - **Green** suggests that everything is on track and progressing well. \n\n2. **RAG (Retrieval-Augmented Generation)**: In the field of artificial intelligence, particularly in natural language processing, RAG refers to a model architecture that combines retrieval-based and generation-based approaches. RAG models first retrieve relevant information from a large corpus of data and then use that information to generate responses. This approach enhances the ability of AI systems to provide accurate and context-rich responses.\n\n

###Structured Output

In [7]:
from typing import List
from pydantic import BaseModel, Field

class MobileReview(BaseModel):
    phone_model: str = Field(description="Name and model of the phone")
    rating: float = Field(description="Overall rating out of 5")
    pros: List[str] = Field(description="List of positive aspects")
    cons: List[str] = Field(description="List of negative aspects")
    summary: str = Field(description="Brief summary of the review")

review_text = """
Just got my hands on the new Galaxy S21 and wow, this thing is slick! The screen is gorgeous,
colors pop like crazy. Camera's insane too, especially at night - my Insta game's never been
stronger. Battery life's solid, lasts me all day no problem.

Not gonna lie though, it's pretty pricey. And what's with ditching the charger? C'mon Samsung.
Also, still getting used to the new button layout, keep hitting Bixby by mistake.

Overall, I'd say it's a solid 4 out of 5. Great phone, but a few annoying quirks keep it from
being perfect. If you're due for an upgrade, definitely worth checking out!
"""

structured_llm = llm.with_structured_output(MobileReview)
output = structured_llm.invoke(review_text)
output

MobileReview(phone_model='Samsung Galaxy S21', rating=4.0, pros=['Gorgeous display with vibrant colors', 'Exceptional camera performance, especially in low light', 'Solid battery life, lasts all day'], cons=['Pricey compared to other models', 'No charger included in the box', 'New button layout can be confusing, often triggering Bixby unintentionally'], summary='The Samsung Galaxy S21 impresses with its stunning display and fantastic camera, making it a great option for upgrades. However, its high price and minor design quirks hold it back from being perfect.')

In [8]:
output.pros

['Gorgeous display with vibrant colors',
 'Exceptional camera performance, especially in low light',
 'Solid battery life, lasts all day']

###Prompt Template

In [9]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("Tell me a short joke about {topic}")
prompt.invoke({"topic": "programming"})

ChatPromptValue(messages=[HumanMessage(content='Tell me a short joke about programming', additional_kwargs={}, response_metadata={})])

In [10]:
chain = prompt | llm | output_parser
chain.invoke({"topic": "programer"})

'Why do programmers prefer dark mode?\n\nBecause light attracts bugs!'

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("Tell me a short joke about {topic}")
llm = ChatOpenAI(model="gpt-4o-mini")
output_parser = StrOutputParser()
chain = prompt | llm | output_parser
result = chain.invoke({"topic": "programming"})
print(result)

Why do programmers prefer dark mode?

Because light attracts bugs!


###LLM Messages

In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

system_message = SystemMessage(content="You are a helpful assistant that tells jokes.")
human_message = HumanMessage(content="Tell me about programming")
llm.invoke([system_message, human_message])

AIMessage(content='Sure! Programming is like telling a computer a story, but instead of fairy tales, you’re giving it instructions to perform tasks. Here’s a joke about programming for you:\n\nWhy do programmers prefer dark mode?\n\nBecause light attracts bugs! 🐞💻\n\nIf you have more questions or want to hear another joke, just let me know!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 24, 'total_tokens': 95, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_44added55e', 'id': 'chatcmpl-BMIg86aUdpcoM26c5TIN8SXNAMcVO', 'finish_reason': 'stop', 'logprobs': None}, id='run-b15f7d4d-346f-44d1-9366-ca2c795db66f-0', usage_metadata={'input_tokens': 24, 'output_tokens': 71, 'total_tokens': 95, 'input_token_d

In [13]:
template = ChatPromptTemplate([
    ("system", "You are a helpful assistant that tells jokes."),
    ("human", "Tell me about {topic}")
])

prompt_value = template.invoke(
    {
        "topic": "programming"
    }
)
prompt_value

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant that tells jokes.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me about programming', additional_kwargs={}, response_metadata={})])

In [14]:
llm.invoke(prompt_value)

AIMessage(content='Sure! Here’s a brief overview of programming and a joke to lighten the mood!\n\nProgramming is the process of designing and building executable computer software to accomplish a specific task. It involves writing code in a programming language, which the computer can understand and execute. Some popular programming languages include Python, Java, JavaScript, C++, and Ruby, among many others.\n\nNow, for the joke:\n\nWhy do programmers prefer dark mode?\n\nBecause light attracts bugs! 🐛💻', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 24, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_44added55e', 'id': 'chatcmpl-BMIg9VcTQfpeKGtWnzZH7xd7WOrjR', 'finish_reason'

In [18]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from typing import List
from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

docx_loader = Docx2txtLoader("C:\Kavita_all_projects\VsCodeAll\Langchain RAG Course 2024\docs\Cybersecurity in the Modern World.docx")
documents = docx_loader.load()

print(len(documents))

splits = text_splitter.split_documents(documents)

print(f"Split the documents into {len(splits)} chunks.")

1
Split the documents into 12 chunks.


In [19]:
documents[0]

Document(metadata={'source': 'C:\\Kavita_all_projects\\VsCodeAll\\Langchain RAG Course 2024\\docs\\Cybersecurity in the Modern World.docx'}, page_content='Cybersecurity in the Modern World\n\nIntroduction In today’s digital age, cybersecurity has become a critical concern for individuals, businesses, and governments. With the increasing reliance on digital technologies and the internet, cyber threats are growing in both volume and sophistication. Cybersecurity is the practice of defending systems, networks, and data from cyber attacks, unauthorized access, and damage. As more data is stored online and new technologies emerge, protecting digital assets has never been more important.\n\nThe Importance of Cybersecurity Cybersecurity plays a crucial role in maintaining the confidentiality, integrity, and availability of information. The importance of cybersecurity can be understood through the following aspects:\n\nProtecting Sensitive Data: Personal information, financial records, and int

In [20]:
splits[1]

Document(metadata={'source': 'C:\\Kavita_all_projects\\VsCodeAll\\Langchain RAG Course 2024\\docs\\Cybersecurity in the Modern World.docx'}, page_content='Protecting Sensitive Data: Personal information, financial records, and intellectual property must be protected from theft or unauthorized access.\n\nEnsuring Operational Continuity: Cyber attacks can disrupt critical business operations, causing significant financial and reputational damage.\n\nMaintaining Trust: Organizations that fail to secure customer data risk losing consumer trust, which can have long-term consequences on their brand and bottom line.\n\nPreventing National Security Threats: Cybersecurity is essential for safeguarding national security, as attacks on critical infrastructure, defense systems, and government agencies can have devastating consequences.\n\nCommon Cybersecurity Threats There are various types of cyber threats that organizations and individuals must defend against. These include:')

In [21]:
splits[0].metadata

{'source': 'C:\\Kavita_all_projects\\VsCodeAll\\Langchain RAG Course 2024\\docs\\Cybersecurity in the Modern World.docx'}

In [22]:
splits[0].page_content

'Cybersecurity in the Modern World\n\nIntroduction In today’s digital age, cybersecurity has become a critical concern for individuals, businesses, and governments. With the increasing reliance on digital technologies and the internet, cyber threats are growing in both volume and sophistication. Cybersecurity is the practice of defending systems, networks, and data from cyber attacks, unauthorized access, and damage. As more data is stored online and new technologies emerge, protecting digital assets has never been more important.\n\nThe Importance of Cybersecurity Cybersecurity plays a crucial role in maintaining the confidentiality, integrity, and availability of information. The importance of cybersecurity can be understood through the following aspects:\n\nProtecting Sensitive Data: Personal information, financial records, and intellectual property must be protected from theft or unauthorized access.'

In [26]:
def load_documents(folder_path: str) -> List[Document]:
    documents = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if filename.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
        elif filename.endswith('.docx'):
            loader = Docx2txtLoader(file_path)
        else:
            print(f"Unsupported file type: {filename}")
            continue
        documents.extend(loader.load())
    return documents

# Load documents from a folder
folder_path = "C:\Kavita_all_projects\VsCodeAll\Langchain RAG Course 2024\content"
documents = load_documents(folder_path)

print(f"Loaded {len(documents)} documents from the folder.")
splits = text_splitter.split_documents(documents)
print(f"Split the documents into {len(splits)} chunks.")

Loaded 5 documents from the folder.
Split the documents into 25 chunks.


In [27]:
embeddings = OpenAIEmbeddings()

# 4. Embedding Documents

document_embeddings = embeddings.embed_documents([split.page_content for split in splits])

print(f"Created embeddings for {len(document_embeddings)} document chunks.")

Created embeddings for 25 document chunks.


In [28]:
document_embeddings[0]

[0.0017408289713785052,
 -0.008500511758029461,
 0.011397300288081169,
 -0.034785956144332886,
 -0.00030142979812808335,
 -0.006262084934860468,
 -0.01780943013727665,
 0.022794600576162338,
 -0.017392978072166443,
 0.0090761948376894,
 -0.002034794772043824,
 0.025844495743513107,
 -0.025917988270521164,
 0.025795502588152885,
 -0.0003175060555804521,
 0.01708676479756832,
 0.036378271877765656,
 0.0036868215538561344,
 0.009229302406311035,
 -0.017197001725435257,
 -0.015004506334662437,
 0.027730777859687805,
 0.005224017892032862,
 0.021190037950873375,
 -0.0056190346367657185,
 -0.0015739420196041465,
 0.009063947014510632,
 -0.029274098575115204,
 0.0015846595633774996,
 -0.009976465255022049,
 0.006032424047589302,
 0.010870611295104027,
 -0.015727171674370766,
 -0.011293187737464905,
 0.0008122337167151272,
 -0.005922186654061079,
 -0.007600241806358099,
 -0.027559297159314156,
 -0.025072835385799408,
 -0.03277719020843506,
 0.02383572980761528,
 -0.005184209905564785,
 0.00888

In [29]:
# !pip install sentence_transformers
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
document_embeddings = embedding_function.embed_documents([split.page_content for split in splits])
document_embeddings[0]

C:\Users\iamka\AppData\Local\Temp\ipykernel_23164\3719339062.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


[0.008057322353124619,
 0.039185114204883575,
 0.12393904477357864,
 0.017161790281534195,
 0.09545053541660309,
 0.046825408935546875,
 -0.03508273884654045,
 -0.0015550898388028145,
 0.04706399887800217,
 0.05612718313932419,
 -0.024742573499679565,
 -0.06114225462079048,
 -0.06893186271190643,
 -0.01962474174797535,
 0.05318549647927284,
 -0.037784870713949203,
 -0.09972814470529556,
 -0.007323076017200947,
 -0.025913681834936142,
 -0.06391186267137527,
 -0.007754736114293337,
 0.06783358752727509,
 -0.01906299777328968,
 0.010067430324852467,
 -0.11095019429922104,
 -0.024519361555576324,
 -0.041749756783246994,
 0.047355253249406815,
 -0.061367496848106384,
 0.10323343425989151,
 -0.01980159804224968,
 0.05256950110197067,
 0.050683874636888504,
 0.04048338532447815,
 -0.05237331613898277,
 0.00653803488239646,
 0.026904698461294174,
 0.0190869253128767,
 -0.05621621385216713,
 0.032021310180425644,
 -0.055383600294589996,
 -0.09858544170856476,
 0.01906222477555275,
 -0.021454961

###Create and persist Chroma vector store

In [32]:
from langchain_chroma import Chroma

embedding_function = OpenAIEmbeddings()
collection_name = "my_collection"
vectorstore = Chroma.from_documents(collection_name=collection_name, documents=splits, embedding=embedding_function, persist_directory="./chroma_db")
#db.persist()

print("Vector store created and persisted to './chroma_db'")

Vector store created and persisted to './chroma_db'


In [33]:
# 5. Perform similarity search

query = "When was GreenGrow Innovations founded?"
search_results = vectorstore.similarity_search(query, k=2)

print(f"\nTop 2 most relevant chunks for the query: '{query}'\n")
for i, result in enumerate(search_results, 1):
    print(f"Result {i}:")
    print(f"Source: {result.metadata.get('source', 'Unknown')}")
    print(f"Content: {result.page_content}")
    print()


Top 2 most relevant chunks for the query: 'When was GreenGrow Innovations founded?'

Result 1:
Source: C:\Kavita_all_projects\VsCodeAll\Langchain RAG Course 2024\content\Company_ GreenFields BioTech.docx
Content: Company: GreenFields BioTech

Headquarters: GreenFields BioTech is headquartered in Zurich, Switzerland. Known for its groundbreaking research in sustainable agriculture and biotechnology, the company has strategically positioned itself in Zurich, a city recognized for its leadership in scientific research and innovation. This location provides GreenFields BioTech with an ideal environment to collaborate with leading academic institutions and industry experts, driving forward its mission to create eco-friendly farming solutions.

Result 2:
Source: C:\Kavita_all_projects\VsCodeAll\Langchain RAG Course 2024\content\Company_ TechWave Innovations.docx
Content: Company: TechWave Innovations

Headquarters: TechWave Innovations is headquartered in San Francisco, California, USA. As 

In [34]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
retriever.invoke("When was GreenGrow Innovations founded?")

[Document(id='d1bfe2f9-f39e-400f-b22d-a1a26f503960', metadata={'source': 'C:\\Kavita_all_projects\\VsCodeAll\\Langchain RAG Course 2024\\content\\Company_ GreenFields BioTech.docx'}, page_content='Company: GreenFields BioTech\n\nHeadquarters: GreenFields BioTech is headquartered in Zurich, Switzerland. Known for its groundbreaking research in sustainable agriculture and biotechnology, the company has strategically positioned itself in Zurich, a city recognized for its leadership in scientific research and innovation. This location provides GreenFields BioTech with an ideal environment to collaborate with leading academic institutions and industry experts, driving forward its mission to create eco-friendly farming solutions.'),
 Document(id='635cb959-d24c-48f2-a195-d2f554125fe2', metadata={'source': 'C:\\Kavita_all_projects\\VsCodeAll\\Langchain RAG Course 2024\\content\\Company_ TechWave Innovations.docx'}, page_content='Company: TechWave Innovations\n\nHeadquarters: TechWave Innovatio

In [35]:
from langchain_core.prompts import ChatPromptTemplate
template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer: """
prompt = ChatPromptTemplate.from_template(template)

In [36]:
from langchain.schema.runnable import RunnablePassthrough
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()} | prompt
)
rag_chain.invoke("When was GreenGrow Innovations founded?")

ChatPromptValue(messages=[HumanMessage(content="Answer the question based only on the following context:\n[Document(id='d1bfe2f9-f39e-400f-b22d-a1a26f503960', metadata={'source': 'C:\\\\Kavita_all_projects\\\\VsCodeAll\\\\Langchain RAG Course 2024\\\\content\\\\Company_ GreenFields BioTech.docx'}, page_content='Company: GreenFields BioTech\\n\\nHeadquarters: GreenFields BioTech is headquartered in Zurich, Switzerland. Known for its groundbreaking research in sustainable agriculture and biotechnology, the company has strategically positioned itself in Zurich, a city recognized for its leadership in scientific research and innovation. This location provides GreenFields BioTech with an ideal environment to collaborate with leading academic institutions and industry experts, driving forward its mission to create eco-friendly farming solutions.'), Document(id='635cb959-d24c-48f2-a195-d2f554125fe2', metadata={'source': 'C:\\\\Kavita_all_projects\\\\VsCodeAll\\\\Langchain RAG Course 2024\\\\c

In [37]:
def docs2str(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [38]:
rag_chain = (
    {"context": retriever | docs2str, "question": RunnablePassthrough()} | prompt
)
rag_chain.invoke("When was GreenGrow Innovations founded?")

ChatPromptValue(messages=[HumanMessage(content='Answer the question based only on the following context:\nCompany: GreenFields BioTech\n\nHeadquarters: GreenFields BioTech is headquartered in Zurich, Switzerland. Known for its groundbreaking research in sustainable agriculture and biotechnology, the company has strategically positioned itself in Zurich, a city recognized for its leadership in scientific research and innovation. This location provides GreenFields BioTech with an ideal environment to collaborate with leading academic institutions and industry experts, driving forward its mission to create eco-friendly farming solutions.\n\nCompany: TechWave Innovations\n\nHeadquarters: TechWave Innovations is headquartered in San Francisco, California, USA. As a leader in cutting-edge AI and machine learning solutions, the company thrives in the heart of Silicon Valley, benefiting from its proximity to tech giants and a dynamic startup ecosystem. With its headquarters in this global tech

In [39]:
rag_chain = (
    {"context": retriever | docs2str, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
question = "When was GreenGrow Innovations founded?"
response = rag_chain.invoke(question)
print(response)

The provided context does not contain information about the founding date of GreenGrow Innovations.


###Conversational RAG

####Handling Follow Up Questions

In [40]:
# Example conversation
from langchain_core.messages import HumanMessage, AIMessage
chat_history = []
chat_history.extend([
    HumanMessage(content=question),
    AIMessage(content=response)
])

In [41]:
chat_history

[HumanMessage(content='When was GreenGrow Innovations founded?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The provided context does not contain information about the founding date of GreenGrow Innovations.', additional_kwargs={}, response_metadata={})]

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)


contextualize_chain = contextualize_q_prompt | llm | StrOutputParser()
contextualize_chain.invoke({"input": "Where it is headquartered?", "chat_history": chat_history})

'What is the location of the headquarters of GreenGrow Innovations?'

In [43]:
from langchain.chains import create_history_aware_retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever.invoke({"input": "Where it is headquartered?", "chat_history": chat_history})

[Document(id='d1bfe2f9-f39e-400f-b22d-a1a26f503960', metadata={'source': 'C:\\Kavita_all_projects\\VsCodeAll\\Langchain RAG Course 2024\\content\\Company_ GreenFields BioTech.docx'}, page_content='Company: GreenFields BioTech\n\nHeadquarters: GreenFields BioTech is headquartered in Zurich, Switzerland. Known for its groundbreaking research in sustainable agriculture and biotechnology, the company has strategically positioned itself in Zurich, a city recognized for its leadership in scientific research and innovation. This location provides GreenFields BioTech with an ideal environment to collaborate with leading academic institutions and industry experts, driving forward its mission to create eco-friendly farming solutions.'),
 Document(id='635cb959-d24c-48f2-a195-d2f554125fe2', metadata={'source': 'C:\\Kavita_all_projects\\VsCodeAll\\Langchain RAG Course 2024\\content\\Company_ TechWave Innovations.docx'}, page_content='Company: TechWave Innovations\n\nHeadquarters: TechWave Innovatio

In [44]:
retriever.invoke("Where it is headquartered?")

[Document(id='b594afb5-f7cb-41bc-bdb8-92de43751840', metadata={'source': 'C:\\Kavita_all_projects\\VsCodeAll\\Langchain RAG Course 2024\\content\\Company_ QuantumNext Systems.docx'}, page_content='Company: QuantumNext Systems\n\nHeadquarters: QuantumNext Systems is headquartered in Bangalore, Karnataka, India. The company, specializing in quantum computing and advanced data processing, is situated in the bustling tech metropolis of Bangalore, often referred to as the "Silicon Valley of India." From this technology capital, QuantumNext Systems is well-positioned to tap into India\'s rich pool of engineering talent and growing tech ecosystem, enabling it to push the boundaries of computational innovation.'),
 Document(id='635cb959-d24c-48f2-a195-d2f554125fe2', metadata={'source': 'C:\\Kavita_all_projects\\VsCodeAll\\Langchain RAG Course 2024\\content\\Company_ TechWave Innovations.docx'}, page_content='Company: TechWave Innovations\n\nHeadquarters: TechWave Innovations is headquartered i

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the following context to answer the user's question."),

    ("system", "Context: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [46]:
rag_chain.invoke({"input": "Where it is headquartered?", "chat_history":chat_history})

{'input': 'Where it is headquartered?',
 'chat_history': [HumanMessage(content='When was GreenGrow Innovations founded?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The provided context does not contain information about the founding date of GreenGrow Innovations.', additional_kwargs={}, response_metadata={})],
 'context': [Document(id='d1bfe2f9-f39e-400f-b22d-a1a26f503960', metadata={'source': 'C:\\Kavita_all_projects\\VsCodeAll\\Langchain RAG Course 2024\\content\\Company_ GreenFields BioTech.docx'}, page_content='Company: GreenFields BioTech\n\nHeadquarters: GreenFields BioTech is headquartered in Zurich, Switzerland. Known for its groundbreaking research in sustainable agriculture and biotechnology, the company has strategically positioned itself in Zurich, a city recognized for its leadership in scientific research and innovation. This location provides GreenFields BioTech with an ideal environment to collaborate with leading academic institutions and indust

###Building Multi User Chatbot

In [ ]:
import sqlite3
from datetime import datetime

DB_NAME = "rag_app.db"

def get_db_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    return conn

def create_application_logs():
    conn = get_db_connection()
    conn.execute('''CREATE TABLE IF NOT EXISTS application_logs
                    (id INTEGER PRIMARY KEY AUTOINCREMENT,
                     session_id TEXT,
                     user_query TEXT,
                     gpt_response TEXT,
                     model TEXT,
                     created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    conn.close()

def insert_application_logs(session_id, user_query, gpt_response, model):
    conn = get_db_connection()
    conn.execute('INSERT INTO application_logs (session_id, user_query, gpt_response, model) VALUES (?, ?, ?, ?)',
                 (session_id, user_query, gpt_response, model))
    conn.commit()
    conn.close()

def get_chat_history(session_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT user_query, gpt_response FROM application_logs WHERE session_id = ? ORDER BY created_at', (session_id,))
    messages = []
    for row in cursor.fetchall():
        messages.extend([
            {"role": "human", "content": row['user_query']},
            {"role": "ai", "content": row['gpt_response']}
        ])
    conn.close()
    return messages

create_application_logs()

In [48]:
import uuid
session_id = str(uuid.uuid4())
chat_history = get_chat_history(session_id)
print(chat_history)
question1 = "When was GreenGrow Innovations founded?"
answer1 = rag_chain.invoke({"input": question1, "chat_history":chat_history})['answer']
insert_application_logs(session_id, question1, answer1, "gpt-4-o-mini")
print(f"Human: {question1}")
print(f"AI: {answer1}\n")

[]
Human: When was GreenGrow Innovations founded?
AI: I don't have information on a company called GreenGrow Innovations. However, I can provide information about GreenFields BioTech, which focuses on sustainable agriculture and is headquartered in Zurich, Switzerland. If you meant GreenFields BioTech or need information on another specific company, please clarify!



In [49]:
question2 = "Where it is headquartered?"
chat_history = get_chat_history(session_id)
print(chat_history)
answer2 = rag_chain.invoke({"input": question2, "chat_history":chat_history})['answer']
insert_application_logs(session_id, question2, answer2, "gpt-3.5-turbo")
print(f"Human: {question2}")
print(f"AI: {answer2}\n")

[{'role': 'human', 'content': 'When was GreenGrow Innovations founded?'}, {'role': 'ai', 'content': "I don't have information on a company called GreenGrow Innovations. However, I can provide information about GreenFields BioTech, which focuses on sustainable agriculture and is headquartered in Zurich, Switzerland. If you meant GreenFields BioTech or need information on another specific company, please clarify!"}]
Human: Where it is headquartered?
AI: GreenFields BioTech is headquartered in Zurich, Switzerland.



New User

In [50]:
session_id = str(uuid.uuid4())
question = "What is GreenGrow"
chat_history = get_chat_history(session_id)
print(chat_history)
answer = rag_chain.invoke({"input": question, "chat_history":chat_history})['answer']
insert_application_logs(session_id, question, answer, "gpt-3.5-turbo")
print(f"Human: {question}")
print(f"AI: {answer}\n")

[]
Human: What is GreenGrow
AI: Based on the provided context, there is no specific mention of "GreenGrow." It could potentially be a different entity, initiative, or product related to agriculture or sustainability. If you're looking for information about a particular organization or program named GreenGrow, please provide more details, and I'll do my best to assist you!

